# **Start Section:**

In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
# The IBUG cells later change directory (%cd ibug, %cd .., %cd ..) and then use
# "./content/drive/..." and "/content/drive/..." (Colab layout); on Kaggle both are linked
# to /kaggle/working so every Excel file lands in the same output folder.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d04_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # water_quality_wqi train
    shutil.copy(glob.glob("/kaggle/input/**/d04_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # water_quality_wqi test
    for _link in ["/content", "/kaggle/content"]:
        if not os.path.exists(_link):
            os.symlink("/kaggle/working", _link)
    print("Kaggle: water_quality_wqi data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: water_quality_wqi data copied to ./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/data


In [2]:
# FIX K1: the treeffuser fork pins numpy<2 / scikit-learn==1.5.0 / lightgbm==4.3.0, which cannot be
# installed on Kaggle's Python 3.13. On Kaggle install the same code without those pins.
if IN_KAGGLE:
    !pip install -q --no-deps git+https://github.com/DaneshSelwal/treeffuser.git
    !pip install -q jaxtyping einops ml-collections
else:
    !pip install git+https://github.com/DaneshSelwal/treeffuser.git


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
treeffuser 0.1.3 requires jaxtyping<0.3.0,>=0.2.19, but you have jaxtyping 0.3.11 which is incompatible.
treeffuser 0.1.3 requires lightgbm==4.3.0, but you have lightgbm 4.6.0 which is incompatible.
treeffuser 0.1.3 requires ml-collections<0.2.0,>=0.1.1, but you have ml-collections 1.1.0 which is incompatible.
treeffuser 0.1.3 requires numpy<2.0,>=1.24, but you have numpy 2.1.3 which is incompatible.
treeffuser 0.1.3 requires scikit-learn==1.5.0, but you have scikit-learn 1.6.1 which is incompatible.


In [3]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel with os._exit(0).
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 mapie==0.6.0 puncc==0.8.0 properscoring==0.1 XlsxWriter==3.2.9
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install mapie==0.6.0 -q
    !pip install puncc==0.8.0 -q
    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ngboost
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from mapie.subsample import Subsample
from mapie.regression import MapieRegressor
from deel.puncc.metrics import regression_sharpness, regression_mean_coverage
from deel.puncc.api.prediction import BasePredictor, DualPredictor
from deel.puncc.regression import SplitCP, CVPlus, CQR
from deel.puncc.plotting import plot_prediction_intervals
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from mapie.metrics import regression_coverage_score
from sklearn.model_selection import KFold
from PIL import Image as PImage
from pathlib import Path
from scipy.stats import norm, kstest
from pytorch_tabnet import TabNetRegressor
from treeffuser import Treeffuser, Samples
from treeffuser._score_models import TreeBasedScoreModel

[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


I0000 00:00:1795405926.211797      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795405944.046656      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [5]:
# Go to find & replace button and replace (water_quality_wqi_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [6]:
feature_names = ['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8', 'R9', 'R10', 'R11', 'R12', 'R13', 'R14', 'R15', 'R16', 'R17', 'R18', 'R19', 'R20', 'R21', 'R22', 'R23', 'R24', 'R25', 'R26', 'R27', 'R28', 'R29', 'R30', 'R31', 'R32', 'R33', 'R34', 'R35', 'R36', 'R37', 'R38', 'R39', 'R40', 'R41', 'R42', 'R43', 'R44', 'R45', 'R46', 'R47', 'R48', 'R49', 'R50', 'R51', 'R52', 'R53', 'R54', 'R55', 'R56', 'R57', 'R58', 'R59', 'R60', 'R61', 'R62', 'R63', 'R64', 'R65', 'R66', 'R67', 'R68', 'R69', 'R70', 'R71', 'R72', 'R73', 'R74', 'R75', 'R76', 'R77', 'R78', 'R79', 'R80', 'R81', 'R82', 'R83', 'R84', 'R85', 'R86', 'R87', 'R88', 'R89', 'R90', 'R91', 'R92', 'R93', 'R94', 'R95', 'R96', 'R97', 'R98', 'R99', 'R100', 'R101', 'R102', 'R103', 'R104', 'R105', 'R106', 'R107', 'R108', 'R109', 'R110', 'R111', 'R112', 'R113', 'R114', 'R115', 'R116', 'R117', 'R118', 'R119', 'R120', 'R121', 'R122', 'R123', 'R124', 'R125', 'R126', 'R127', 'R128', 'R129', 'R130', 'R131', 'R132', 'R133', 'R134', 'R135', 'R136', 'R137', 'R138', 'R139', 'R140', 'R141', 'R142', 'R143', 'R144', 'R145', 'R146', 'R147', 'R148', 'R149', 'R150', 'R151']
TARGET_LABEL = 'WQI'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)

In [7]:
train_data_path = "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [8]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (190, 152)
First 5 rows of training data:
        R1      R2      R3      R4      R5      R6      R7      R8      R9  \
0  3.5960  3.6080  3.0100  3.6130  3.3760  2.2780  1.3410  3.4570  3.3540   
1  2.9540  3.0470  3.1160  4.2640  4.4090  3.4810  2.3760  3.4030  3.4320   
2  2.4870  2.9430  2.5290  3.7280  2.9870  1.8890  1.9960  2.1020  1.6180   
3  1.3640  1.4186  1.3789  1.4776  1.5063  1.4047  1.2648  1.5166  1.5443   
4  1.3074  1.3820  1.3476  1.3986  1.3403  1.3004  1.3719  1.3984  1.3865   

      R10  ...    R143    R144    R145    R146    R147   R148    R149    R150  \
0  1.9900  ...  2.7930  1.9660  2.3070  1.5280  1.9380  3.371  2.2060  1.5220   
1  2.6350  ...  3.0260  2.3530  2.0400  1.7400  2.6250  4.026  2.9010  1.7450   
2  1.2720  ...  1.2760  1.5990  1.8610  1.0290  1.8690  3.905  2.3200  1.3310   
3  1.3834  ...  1.2223  1.0919  1.0995  1.0856  1.1728  1.315  1.2341  1.0753   
4  1.3964  ...  1.1691  1.0689  1.1230  1.1384  1.1810  1.235  1

In [9]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (190, 151)
Shape of y_train: (190,)
Shape of X_test: (82, 151)
Shape of y_test: (82,)


In [10]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[ 1.22103295  1.13089815  0.61081314  0.73682842  0.54673262 -0.0261968
  -0.51531151  0.70303503  0.5510272  -0.2441054   0.17648144  0.69219002
   0.9485328   0.82481901  0.34566436 -0.40609522 -0.463771    0.43872889
   0.89421381  1.05057715  1.04529849  1.03900179  0.42575888  0.173456
   0.45642529  0.83037026  0.89010153  0.77972274  0.71866378  0.6206252
   0.82167079  0.97412629  0.67298061  0.39758814  0.52679374  0.94375937
   0.74197071  0.86577935  1.27064247  1.51437415  0.46596778 -0.30888056
  -0.12461695  0.49591603  0.72957494  0.5811918   0.31848773  0.53466849
   0.20246884  0.11900459  0.99635035  0.72105768  0.43065398  0.36777539
   0.10638109 -0.0263905   0.2318847   0.82337505  0.74363607  0.85248236
   0.71669043 -0.11909161 -0.10843386  0.51688952  0.84551287 -0.1576873
  -0.12716017  0.37884167  0.35396679 -0.24556803  0.07962918  0.82628894
   0.54494889  1.02963853  1.17379609  0.60350175 -0.17692428 -0.31350223
   

# **Functions:**

In [11]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [12]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']

In [13]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_plots(predictions_df, y_test, excel_path):
    # 1. Predicted vs. Actual Plot
    plt.figure(figsize=(12, 8))
    sns.scatterplot(x='Actual', y='Mean', data=predictions_df)
    plt.plot([predictions_df['Actual'].min(), predictions_df['Actual'].max()],
             [predictions_df['Actual'].min(), predictions_df['Actual'].max()],
             color='red', linestyle='--')
    plt.title(f'Predicted {TARGET_LABEL} vs. Actual {TARGET_LABEL}')
    plt.xlabel(f'Actual {TARGET_LABEL}')
    plt.ylabel(f'Predicted Mean {TARGET_LABEL}')
    plt.tight_layout()
    predicted_vs_actual_path = 'predicted_vs_actual.png'
    _ensure_parent_dir(predicted_vs_actual_path)
    plt.savefig(_ensure_parent_dir(predicted_vs_actual_path))
    plt.close()

    # 2. Uncertainty Bands
    plt.figure(figsize=(12, 8))
    plt.errorbar(x=range(len(predictions_df)), y=predictions_df['Mean'],
                 yerr=predictions_df['StdDev'], fmt='o', markersize=4, ecolor='red', elinewidth=1, capsize=5, alpha=0.7)
    plt.plot(range(len(predictions_df)), predictions_df['Mean'], color='black', linewidth=0.5, alpha=0.7)
    plt.title('Predicted Mean with Uncertainty Bands', fontsize=16)
    plt.xlabel('Sample Number', fontsize=14)
    plt.ylabel(f'Predicted Mean {TARGET_LABEL}')
    plt.tight_layout()
    uncertainty_bands_path = 'uncertainty_bands.png'
    _ensure_parent_dir(uncertainty_bands_path)
    plt.savefig(_ensure_parent_dir(uncertainty_bands_path))
    plt.close()

    # 3. Histogram of Standard Deviations
    plt.figure(figsize=(12, 8))
    sns.histplot(predictions_df['StdDev'], bins=20, kde=True)
    plt.title('Distribution of Predicted Standard Deviations')
    plt.xlabel('Standard Deviation')
    plt.ylabel('Frequency')
    plt.tight_layout()
    histogram_stddev_path = 'histogram_stddev.png'
    _ensure_parent_dir(histogram_stddev_path)
    plt.savefig(_ensure_parent_dir(histogram_stddev_path))
    plt.close()

    # 4. Residual Plot
    residuals = predictions_df['Actual'] - predictions_df['Mean']
    plt.figure(figsize=(12, 8))
    sns.scatterplot(x=predictions_df['Mean'], y=residuals)
    plt.axhline(0, color='red', linestyle='--')
    plt.title('Residual Plot')
    plt.xlabel(f'Predicted Mean {TARGET_LABEL}')
    plt.ylabel('Residuals')
    plt.tight_layout()
    residual_plot_path = 'residual_plot.png'
    _ensure_parent_dir(residual_plot_path)
    plt.savefig(_ensure_parent_dir(residual_plot_path))
    plt.close()

    # Plot the standard deviation of the predictions
    plt.figure(figsize=(12, 6))
    plt.plot(
        predictions_df['StdDev'],
        label='Standard Deviation',
        color='green',
        marker='o',
        linestyle='dashed'
    )
    plt.xlabel('Sample Number')
    plt.ylabel('Standard Deviation')
    plt.title('Standard Deviation of Predictions')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()
    stddev_plot_path = 'stddev_plot.png'
    _ensure_parent_dir(stddev_plot_path)
    plt.savefig(_ensure_parent_dir(stddev_plot_path))
    plt.close()

    # Plot the actual vs predicted mean with confidence intervals
    plt.figure(figsize=(12, 6))
    plt.plot(
        y_test,
        label=f'Actual {TARGET_LABEL}',
        color='black',
        marker='o',
        linestyle='dashed',
        linewidth=1,
        markersize=4
    )
    plt.plot(
        predictions_df['Mean'],
        label=f'Predicted Mean {TARGET_LABEL}',
        color='red',
        marker='x',
        linestyle='dashed',
        linewidth=1,
        markersize=4
    )
    confidence_interval = 1.96 * predictions_df['StdDev']
    plt.fill_between(
        np.arange(len(predictions_df)),
        predictions_df['Mean'] - confidence_interval,
        predictions_df['Mean'] + confidence_interval,
        color='blue',
        alpha=0.2,
        label='95% Confidence Interval'
    )
    plt.xlabel('Sample Number')
    plt.ylabel(f' {TARGET_LABEL}')
    plt.title('Predictions with Confidence Intervals')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()
    prediction_plot_path = 'prediction_plot.png'
    _ensure_parent_dir(prediction_plot_path)
    plt.savefig(_ensure_parent_dir(prediction_plot_path))
    plt.close()

    # Create a Pandas Excel writer using XlsxWriter as the engine.
    writer = pd.ExcelWriter(_ensure_parent_dir(excel_path), engine='xlsxwriter')

    # Write the dataframe to a worksheet.
    writer
    predictions_df.to_excel(writer, sheet_name='Predictions', index=False)

    # Access the XlsxWriter workbook and worksheet objects from the writer object.
    workbook = writer.book

    # Insert the plots into new worksheets.
    worksheet_predicted_vs_actual = workbook.add_worksheet('Predicted vs Actual')
    worksheet_predicted_vs_actual.insert_image('A1', predicted_vs_actual_path)

    worksheet_uncertainty_bands = workbook.add_worksheet('Uncertainty Bands')
    worksheet_uncertainty_bands.insert_image('A1', uncertainty_bands_path)

    worksheet_histogram_stddev = workbook.add_worksheet('Histogram StdDev')
    worksheet_histogram_stddev.insert_image('A1', histogram_stddev_path)

    worksheet_residual_plot = workbook.add_worksheet('Residual Plot')
    worksheet_residual_plot.insert_image('A1', residual_plot_path)

    worksheet_stddev = workbook.add_worksheet('StdDev Plot')
    worksheet_stddev.insert_image('A1', stddev_plot_path)

    worksheet_prediction = workbook.add_worksheet('Predictions Plot')
    worksheet_prediction.insert_image('A1', prediction_plot_path)

    # Close the Pandas Excel writer and output the Excel file.
    writer.close()
    print(f"Excel file saved at {excel_path}")

    # Optionally, remove the image files now that they're embedded in the Excel file
    if os.path.exists(str(predicted_vs_actual_path)): os.remove(str(predicted_vs_actual_path))
    if os.path.exists(str(uncertainty_bands_path)): os.remove(str(uncertainty_bands_path))
    if os.path.exists(str(histogram_stddev_path)): os.remove(str(histogram_stddev_path))
    if os.path.exists(str(residual_plot_path)): os.remove(str(residual_plot_path))
    if os.path.exists(str(stddev_plot_path)): os.remove(str(stddev_plot_path))
    if os.path.exists(str(prediction_plot_path)): os.remove(str(prediction_plot_path))

In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_crps_decomposition(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_bins=N_BINS
):

    # -------------------------------
    # Input handling
    # -------------------------------
    mu_pred = predictions_df['Mean'].values
    sigma_pred = predictions_df['StdDev'].values

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()

    # -------------------------------
    # Total CRPS
    # -------------------------------
    crps_values = ps.crps_gaussian(
        y_true, mu=mu_pred, sig=sigma_pred
    )
    total_crps = np.mean(crps_values)

    # -------------------------------
    # FIX P1: Reliability / Resolution / Uncertainty from the Hersbach (2000) decomposition,
    # "Decomposition of the CRPS for ensemble prediction systems", Weather and Forecasting 15.
    # Was: Reliability = mean of the model's own CRPS over 10 groups (= total CRPS), so
    # Resolution always came out = Uncertainty and the split carried no information.
    # Each predicted Gaussian is represented by M members at its quantile levels (k-0.5)/M.
    #   CRPS_ens = Reliability + CRPS_pot,  CRPS_pot = Uncertainty - Resolution
    # -------------------------------
    M = 100
    sig_safe = np.clip(sigma_pred, 1e-12, None)
    levels = (np.arange(1, M + 1) - 0.5) / M
    ens = np.sort(norm.ppf(levels[None, :], loc=mu_pred[:, None], scale=sig_safe[:, None]), axis=1)  # (n, M)
    yy = y_true[:, None]
    n_obs = len(y_true)

    # alpha_i, beta_i for the inner bins [x_i, x_{i+1}], i = 1..M-1 (Hersbach eq. 26)
    lo, hi = ens[:, :-1], ens[:, 1:]
    alpha_in = np.where(yy > hi, hi - lo, np.where(yy > lo, yy - lo, 0.0))
    beta_in = np.where(yy < lo, hi - lo, np.where(yy < hi, hi - yy, 0.0))
    alpha_bar = np.zeros(M + 1)
    beta_bar = np.zeros(M + 1)
    alpha_bar[1:M] = alpha_in.mean(axis=0)
    beta_bar[1:M] = beta_in.mean(axis=0)
    # outliers (Hersbach eq. 27): below the lowest / above the highest member
    beta_bar[0] = np.mean(np.where(y_true < ens[:, 0], ens[:, 0] - y_true, 0.0))
    alpha_bar[M] = np.mean(np.where(y_true > ens[:, -1], y_true - ens[:, -1], 0.0))

    p = np.arange(M + 1) / M
    g = alpha_bar + beta_bar
    o = np.divide(beta_bar, g, out=np.zeros_like(g), where=g > 0)
    # outlier bins: o_0 = freq(y < x_1), o_M = 1 - freq(y > x_M) (Hersbach eq. 33)
    o[0] = np.mean(y_true < ens[:, 0])
    o[M] = 1.0 - np.mean(y_true > ens[:, -1])
    g[0] = beta_bar[0] / o[0] if o[0] > 0 else 0.0
    g[M] = alpha_bar[M] / (1.0 - o[M]) if o[M] < 1 else 0.0

    reliability = float(np.sum(g * (o - p) ** 2))
    crps_pot = float(np.sum(g * o * (1.0 - o)))
    # Uncertainty = CRPS of the sample climatology of the observations
    uncertainty = float(np.abs(y_true[:, None] - y_true[None, :]).sum() / (2.0 * n_obs ** 2))
    resolution = uncertainty - crps_pot

    # -------------------------------
    # Per-bin table (unchanged): test rows sorted by predicted mean, split into n_bins groups
    # -------------------------------
    sort_idx = np.argsort(mu_pred)
    y_sorted = y_true[sort_idx]
    mu_sorted = mu_pred[sort_idx]
    sigma_sorted = sigma_pred[sort_idx]

    bins = np.array_split(np.arange(len(y_true)), n_bins)

    bin_results = []

    for i, b in enumerate(bins):
        bin_y = y_sorted[b]
        bin_mu = mu_sorted[b]
        bin_sigma = sigma_sorted[b]

        bin_crps = np.mean(
            ps.crps_gaussian(bin_y, mu=bin_mu, sig=bin_sigma)
        )

        bin_results.append({
            "Bin": i + 1,
            "Bin_Size": len(b),
            "Mean_Pred_Mu": np.mean(bin_mu),
            "Mean_Pred_Sigma": np.mean(bin_sigma),
            "Bin_CRPS": bin_crps
        })

    # -------------------------------
    # Summary table
    # -------------------------------
    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Total_CRPS": total_crps,
        "Reliability": reliability,
        "Resolution": resolution,
        "Uncertainty": uncertainty,
        "Num_Bins": n_bins
    }])

    bin_df = pd.DataFrame(bin_results)

    # -------------------------------
    # Save to Excel
    # -------------------------------
    excel_path = Path(excel_path)
    excel_path.parent.mkdir(parents=True, exist_ok=True)

    sheet_summary = f"CRPS_Summary_{model_name}"
    sheet_bins = f"CRPS_Bins_{model_name}"

    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(
        excel_path)),
        engine="openpyxl",
        mode="a" if excel_path.exists() else "w",
        if_sheet_exists="replace"
    ) as writer:
        writer
        summary_df.to_excel(writer, sheet_name=sheet_summary, index=False)
        writer
        bin_df.to_excel(writer, sheet_name=sheet_bins, index=False)

    print(f"✅ CRPS decomposition saved to:\n{excel_path}")
    print(f"   Sheets: {sheet_summary}, {sheet_bins}")

    return summary_df

In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_pit_results(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_bins=N_BINS
):
    """
    Generate PIT values, histogram, calibration diagnostics,
    and save everything to Excel with the PIT plot EMBEDDED.
    """

    # -------------------------------
    # Input handling
    # -------------------------------
    mu_pred = predictions_df['Mean'].values
    sigma_pred = predictions_df['StdDev'].values

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()

    sigma_pred = np.clip(sigma_pred, 1e-6, None)

    # -------------------------------
    # Compute PIT values
    # -------------------------------
    pit_values = norm.cdf(y_true, loc=mu_pred, scale=sigma_pred)

    pit_df = pd.DataFrame({"PIT_Value": pit_values})

    # -------------------------------
    # Histogram data
    # -------------------------------
    hist_density, bin_edges = np.histogram(
        pit_values, bins=n_bins, range=(0, 1), density=True
    )
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    hist_df = pd.DataFrame({
        "Bin_Center": bin_centers,
        "Density": hist_density
    })

    # -------------------------------
    # Calibration diagnostics
    # -------------------------------
    pit_mean = pit_values.mean()
    pit_var = pit_values.var()
    ks_stat, ks_pvalue = kstest(pit_values, 'uniform')

    if ks_pvalue > 0.05:
        diagnosis = "Well-calibrated (Uniform PIT)"
    elif pit_var < 1/12:
        diagnosis = "Over-dispersed (Under-confident)"
    elif pit_var > 1/12:
        diagnosis = "Under-dispersed (Over-confident)"
    else:
        diagnosis = "Biased or miscalibrated"

    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Mean_PIT": pit_mean,
        "Variance_PIT": pit_var,
        "Ideal_Variance_(1/12)": 1/12,
        "KS_Statistic": ks_stat,
        "KS_pvalue": ks_pvalue,
        "Calibration_Diagnosis": diagnosis
    }])

    # -------------------------------
    # Create PIT plot in memory
    # -------------------------------
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(
        pit_values,
        bins=n_bins,
        density=True,
        color="skyblue",
        edgecolor="black",
        alpha=0.75
    )
    ax.axhline(1, color="red", linestyle="--", label="Perfect Calibration")
    ax.set_xlabel("PIT Value")
    ax.set_ylabel("Density")
    ax.set_title(f"PIT Histogram – {model_name}")
    ax.legend()
    fig.tight_layout()

    img_buffer = BytesIO()
    fig.savefig(img_buffer, format="png", dpi=300)
    plt.close(fig)
    img_buffer.seek(0)

    # -------------------------------
    # Save to Excel
    # -------------------------------
    excel_path = Path(excel_path)
    excel_path.parent.mkdir(parents=True, exist_ok=True)

    sheet_values = f"PIT_Values_{model_name}"
    sheet_hist = f"PIT_Histogram_{model_name}"
    sheet_summary = f"PIT_Summary_{model_name}"
    sheet_plot = f"PIT_Plot_{model_name}"

    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(
        excel_path)),
        engine="openpyxl",
        mode="a" if excel_path.exists() else "w",
        if_sheet_exists="replace"
    ) as writer:

        writer
        pit_df.to_excel(writer, sheet_name=sheet_values, index=False)
        writer
        hist_df.to_excel(writer, sheet_name=sheet_hist, index=False)
        writer
        summary_df.to_excel(writer, sheet_name=sheet_summary, index=False)

        # Create empty sheet for plot
        workbook = writer.book
        worksheet = workbook.create_sheet(sheet_plot)

        img = openpyxlImage(img_buffer)
        img.width = img.width * 0.4
        img.height = img.height * 0.4

        img.anchor = "B2"
        worksheet.add_image(img)

    print(f"✅ PIT values, histogram, summary, and EMBEDDED plot saved to:")
    print(f"   {excel_path}")

    return pit_df

In [16]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_calibration_curve(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_quantiles=10
):
    """
    Generate quantile calibration curve (reliability diagram),
    compute calibration errors, and save everything to Excel
    with the plot EMBEDDED.
    """

    # -------------------------------
    # Input handling
    # -------------------------------
    mu_pred = predictions_df['Mean'].values
    sigma_pred = predictions_df['StdDev'].values

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()
    sigma_pred = np.clip(sigma_pred, 1e-6, None)

    # -------------------------------
    # Quantiles
    # -------------------------------
    quantiles = np.linspace(0, 1, n_quantiles + 1)

    observed_freq = []
    predicted_q = []

    for q in quantiles:
        # Predicted value at quantile q
        thresh = norm.ppf(q, loc=mu_pred, scale=sigma_pred)

        # Observed frequency
        obs = np.mean(y_true <= thresh)

        predicted_q.append(q)
        observed_freq.append(obs)

    predicted_q = np.array(predicted_q)
    observed_freq = np.array(observed_freq)

    # -------------------------------
    # Calibration metrics
    # -------------------------------
    mce = np.mean(np.abs(observed_freq - predicted_q))
    rmsce = np.sqrt(np.mean((observed_freq - predicted_q) ** 2))

    calib_df = pd.DataFrame({
        "Predicted_Quantile": predicted_q,
        "Observed_Frequency": observed_freq,
        "Absolute_Error": np.abs(observed_freq - predicted_q)
    })

    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Mean_Calibration_Error (MCE)": mce,
        "RMS_Calibration_Error (RMSCE)": rmsce,
        "Num_Quantiles": n_quantiles
    }])

    # -------------------------------
    # Create calibration plot
    # -------------------------------
    fig, ax = plt.subplots(figsize=(7, 7))

    ax.plot([0, 1], [0, 1], 'k--', label="Perfect Calibration")
    ax.plot(
        predicted_q,
        observed_freq,
        'o-',
        color='teal',
        label=f"{model_name}"
    )

    ax.set_xlabel("Predicted Quantile (Confidence Level)")
    ax.set_ylabel("Observed Frequency")
    ax.set_title(f"Quantile Calibration Curve – {model_name}")
    ax.legend()
    fig.tight_layout()

    img_buffer = BytesIO()
    fig.savefig(img_buffer, format="png", dpi=300)
    plt.close(fig)
    img_buffer.seek(0)

    # -------------------------------
    # Save to Excel
    # -------------------------------
    excel_path = Path(excel_path)
    excel_path.parent.mkdir(parents=True, exist_ok=True)

    sheet_curve = f"Calibration_Curve_{model_name}"
    sheet_summary = f"Calibration_Summary_{model_name}"
    sheet_plot = f"Calibration_Plot_{model_name}"

    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(
        excel_path)),
        engine="openpyxl",
        mode="a" if excel_path.exists() else "w",
        if_sheet_exists="replace"
    ) as writer:

        writer
        calib_df.to_excel(writer, sheet_name=sheet_curve, index=False)
        writer
        summary_df.to_excel(writer, sheet_name=sheet_summary, index=False)

        workbook = writer.book

        worksheet = workbook.create_sheet(sheet_plot)

        img = openpyxlImage(img_buffer)
        img.width = img.width * 0.4
        img.height = img.height * 0.4

        img.anchor = "B2"
        worksheet.add_image(img)

    print("✅ Calibration curve & metrics saved to Excel:")
    print(f"   {excel_path}")

    return calib_df

In [17]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
# Define the path to the folder
folder_path = "/content/drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis"

# Create the folder if it doesn't exist
os.makedirs(folder_path, exist_ok=True)

def plot_metrics_and_save_to_excel(writer, predictions_df, y_true, model_name):
    # Calculate metrics
    mean_pred = predictions_df['Mean']
    std_pred = predictions_df['StdDev']

    # Sharpness is the standard deviation
    sharpness = std_pred.mean()

    # CRPS for each prediction
    crps_values = ps.crps_gaussian(y_true, mean_pred, std_pred)

    # Negative Log Likelihood (NLL) for each prediction
    nll_values = -norm.logpdf(y_true, loc=mean_pred, scale=std_pred)
    nll = nll_values.mean()

    # Calibration (Validity)
    percentiles = np.linspace(0, 100, 101)
    calibration = [np.mean((y_true <= np.percentile(mean_pred + std_pred * np.random.randn(*mean_pred.shape), p))) for p in percentiles]

    # Create a DataFrame to store the metrics
    metrics_df = pd.DataFrame({
        'Mean Prediction': mean_pred,
        'Standard Deviation': std_pred,
        'CRPS': crps_values,
        'NLL': nll_values
    })

    # Add summary statistics
    summary_df = pd.DataFrame({
        'Metric': ['Sharpness', 'Mean CRPS', 'Mean NLL'],
        'Value': [sharpness, crps_values.mean(), nll]
    })

    # Write the metrics and summary to the Excel sheet
    metrics_sheet_name = f'{model_name} Metrics'
    writer
    metrics_df.to_excel(writer, sheet_name=metrics_sheet_name, index=False)

    summary_sheet_name = f'{model_name} Summary'
    writer
    summary_df.to_excel(writer, sheet_name=summary_sheet_name, index=False)

    # Plotting
    fig, axs = plt.subplots(2, 2, figsize=(12, 10))

    # Sharpness plot
    axs[0, 0].hist(std_pred, bins=30, color='skyblue', edgecolor='black')
    axs[0, 0].set_title(f'Sharpness (StdDev) for {model_name}')
    axs[0, 0].set_xlabel('Standard Deviation')
    axs[0, 0].set_ylabel('Frequency')

    # CRPS plot (distribution)
    axs[0, 1].hist(crps_values, bins=30, color='lightgreen', edgecolor='black')
    axs[0, 1].set_title(f'CRPS Distribution for {model_name}')
    axs[0, 1].set_xlabel('CRPS')
    axs[0, 1].set_ylabel('Frequency')

    # NLL plot (distribution)
    axs[1, 0].hist(nll_values, bins=30, color='salmon', edgecolor='black')
    axs[1, 0].set_title(f'NLL Distribution for {model_name}')
    axs[1, 0].set_xlabel('NLL')
    axs[1, 0].set_ylabel('Frequency')

    # Calibration plot
    axs[1, 1].plot(percentiles, calibration, label='Calibration Curve')
    axs[1, 1].plot(percentiles, percentiles / 100, linestyle='--', label='Perfect Calibration')
    axs[1, 1].set_title(f'Calibration (Validity) for {model_name}')
    axs[1, 1].set_xlabel('Predicted Percentile')
    axs[1, 1].set_ylabel('Observed Frequency')
    axs[1, 1].legend()

    plt.tight_layout()

    # Save plots to images
    plot_image_path = f'{model_name}_plots.png'
    fig.savefig(_ensure_parent_dir(plot_image_path))
    plt.close(fig)

    # Insert the plot image into the Excel sheet
    sheet_name = f'{model_name} Plots'
    if sheet_name not in writer.sheets:  # Check if sheet already exists
        # If sheet doesn't exist, create a new one
        worksheet = writer.book.add_worksheet(sheet_name)
    else:
        # If sheet exists, get the existing sheet object to avoid duplicates
        worksheet = writer.sheets[sheet_name]
    worksheet.insert_image('A1', plot_image_path)

# **Tuned Parameters**

In [18]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/datasets/daneshselwal/water-quality-wqi-tuned-params/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


In [19]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 869.3671877788987,
  'cv_rmse': 29.48503328434443,
  'best_params': {'n_estimators': 200,
   'criterion': 'friedman_mse',
   'max_depth': None,
   'min_samples_split': 0.01,
   'min_samples_leaf': 3,
   'min_weight_fraction_leaf': 0.01,
   'max_features': 1.0,
   'max_leaf_nodes': None,
   'min_impurity_decrease': 0.2,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.1},
  'test_mse': 1071.6124042838674,
  'test_rmse': np.float64(32.735491508206614),
  'test_corr_coef': np.float64(0.9307870367802251),
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 869.3671877788987,
  'cv_rmse': 29.48503328434443,
  'best_params': {'n_estimators': 200,
   'criterion': 'friedman_mse',
   'max_depth': None,
   'min_samples_split': 0.01,
   'min_samples_leaf': 3,
   'min_weight_fraction_leaf': 0.01,
   'max_features': 1.0,
   'max_leaf_nodes': None,
   'min_impurity_decrease':

In [20]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 869.3671877788987,
  'cv_rmse': 29.48503328434443,
  'best_params': {'n_estimators': 200,
   'criterion': 'friedman_mse',
   'max_depth': None,
   'min_samples_split': 0.01,
   'min_samples_leaf': 3,
   'min_weight_fraction_leaf': 0.01,
   'max_features': 1.0,
   'max_leaf_nodes': None,
   'min_impurity_decrease': 0.2,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.1},
  'test_mse': 1071.6124042838674,
  'test_rmse': np.float64(32.735491508206614),
  'test_corr_coef': np.float64(0.9307870367802251),
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 869.3671877788987,
  'cv_rmse': 29.48503328434443,
  'best_params': {'n_estimators': 200,
   'criterion': 'friedman_mse',
   'max_depth': None,
   'min_samples_split': 0.01,
   'min_samples_leaf': 3,
   'min_weight_fraction_leaf': 0.01,
   'max_features': 1.0,
   'max_leaf_nodes': None,
   'min_impurity_decrease':

# **Probabilistic distributions using NGBoost**

In [21]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize NGBoost model with best_params
ngb_model = NGBRegressor(Dist=Normal, **best_params)

# Fit the model
ngb_model.fit(X_train, y_train)

# Predict the distribution
pred_dist = ngb_model.pred_dist(X_test)

# Extract mean and standard deviation
predictions_NGB_df = pd.DataFrame({
    'Mean': pred_dist.loc,
    'StdDev': pred_dist.scale,
    'Actual': y_test.ravel()
})

# Calculate and print the mean squared error
mse = mean_squared_error(y_test, pred_dist.loc)
print(f"Mean Squared Error: {mse:.2f}")

# Print the DataFrame with mean and standard deviation
print(predictions_NGB_df.head())

Mean Squared Error: 859.66
         Mean    StdDev  Actual
0  295.085884  0.322187     315
1  209.843743  0.230569     167
2  435.308723  0.192169     516
3  218.288358  0.240409     144
4  134.702255  0.132116     127


In [22]:
generate_and_save_plots(predictions_NGB_df, y_test, "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx")

Excel file saved at ./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx


In [23]:
generate_and_save_crps_decomposition(
    predictions_NGB_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx",
    model_name="NGBoost",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx
   Sheets: CRPS_Summary_NGBoost, CRPS_Bins_NGBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,NGBoost,22.411176,17.040988,32.119836,37.490482,8


In [24]:
generate_and_save_pit_results(
    predictions_NGB_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx",
    model_name="NGBoost",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx


,PIT_Value
0,1.000000
1,0.000000
2,1.000000
3,0.000000
4,0.000000
...,...
77,0.000000
78,0.000000
79,0.000000
80,0.000000


In [25]:
generate_and_save_calibration_curve(
    predictions_NGB_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx",
    model_name="NGBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.841463,0.741463
2,0.2,0.853659,0.653659
3,0.3,0.853659,0.553659
4,0.4,0.853659,0.453659
5,0.5,0.853659,0.353659
6,0.6,0.853659,0.253659
7,0.7,0.853659,0.153659
8,0.8,0.853659,0.053659
9,0.9,0.853659,0.046341


# **Probabilistic distributions using PGBM**

In [26]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
# Note: Adjust the incompatible keys based on PGBM's requirements
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()

def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)
# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params = best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Extract mean and standard deviation
# Note: PGBM might return predictions in a different format, adjust accordingly
predictions_PGBM_df = pd.DataFrame({
    'Mean': pred_dist.mean(axis=0),
    'StdDev': pred_dist.std(axis=0),
    'Actual': y_test.ravel()
})

# Calculate and print the mean squared error
mse = mean_squared_error(y_test, predictions_PGBM_df['Mean'])
print(f"Mean Squared Error: {mse:.2f}")

# Print the DataFrame with mean and standard deviation
print(predictions_PGBM_df.head())

Training on CPU


Estimator 0/300, Train metric: 78.9001
Estimator 1/300, Train metric: 76.0510
Estimator 2/300, Train metric: 73.3512
Estimator 3/300, Train metric: 70.6507
Estimator 4/300, Train metric: 68.2775
Estimator 5/300, Train metric: 65.8602


Estimator 6/300, Train metric: 63.4633
Estimator 7/300, Train metric: 61.1513
Estimator 8/300, Train metric: 59.1168
Estimator 9/300, Train metric: 57.0578
Estimator 10/300, Train metric: 55.1212
Estimator 11/300, Train metric: 53.2731
Estimator 12/300, Train metric: 51.4449
Estimator 13/300, Train metric: 49.6976


Estimator 14/300, Train metric: 48.0189
Estimator 15/300, Train metric: 46.4079
Estimator 16/300, Train metric: 44.8642
Estimator 17/300, Train metric: 43.4459
Estimator 18/300, Train metric: 42.0784
Estimator 19/300, Train metric: 40.7679
Estimator 20/300, Train metric: 39.4945


Estimator 21/300, Train metric: 38.2714
Estimator 22/300, Train metric: 37.0826
Estimator 23/300, Train metric: 35.9078
Estimator 24/300, Train metric: 34.7604
Estimator 25/300, Train metric: 33.7132


Estimator 26/300, Train metric: 32.6903
Estimator 27/300, Train metric: 31.6972
Estimator 28/300, Train metric: 30.7656
Estimator 29/300, Train metric: 29.8942
Estimator 30/300, Train metric: 29.0440
Estimator 31/300, Train metric: 28.2583
Estimator 32/300, Train metric: 27.4161


... (54 outputs left out)

Estimator 285/300, Train metric: 1.5957
Estimator 286/300, Train metric: 1.5873
Estimator 287/300, Train metric: 1.5789
Estimator 288/300, Train metric: 1.5702
Estimator 289/300, Train metric: 1.5613
Estimator 290/300, Train metric: 1.5522
Estimator 291/300, Train metric: 1.5438


Estimator 292/300, Train metric: 1.5344
Estimator 293/300, Train metric: 1.5256
Estimator 294/300, Train metric: 1.5180
Estimator 295/300, Train metric: 1.5104
Estimator 296/300, Train metric: 1.5022
Estimator 297/300, Train metric: 1.4947
Estimator 298/300, Train metric: 1.4865


Estimator 299/300, Train metric: 1.4789


Mean Squared Error: 860.40
         Mean     StdDev  Actual
0  291.942566  11.235856     315
1  181.325562   2.601217     167
2  429.463348  16.016714     516
3  221.204361   3.788760     144
4  133.217651   2.526813     127


In [27]:
generate_and_save_plots(predictions_PGBM_df, y_test, "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx")

Excel file saved at ./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx


In [28]:
generate_and_save_crps_decomposition(
    predictions_PGBM_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx",
    model_name="PGBM",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx
   Sheets: CRPS_Summary_PGBM, CRPS_Bins_PGBM


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,PGBM,20.521602,10.524602,27.487675,37.490482,8


In [29]:
generate_and_save_pit_results(
    predictions_PGBM_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx",
    model_name="PGBM",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx


,PIT_Value
0,9.799215e-01
1,1.822376e-08
2,1.000000e+00
3,1.332078e-92
4,6.933896e-03
...,...
77,7.188269e-09
78,1.064275e-16
79,2.127838e-82
80,6.689463e-04


In [30]:
generate_and_save_calibration_curve(
    predictions_PGBM_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx",
    model_name="PGBM",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.756098,0.656098
2,0.2,0.780488,0.580488
3,0.3,0.804878,0.504878
4,0.4,0.804878,0.404878
5,0.5,0.817073,0.317073
6,0.6,0.829268,0.229268
7,0.7,0.829268,0.129268
8,0.8,0.841463,0.041463
9,0.9,0.865854,0.034146


# **Probabilistic Distribution using GPBoost (Treeffuser)**


In [31]:
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')

gpboost_score_engine = TreeBasedScoreModel(
    model_name='gpboost',
    model_params=best_params,
    seed=42,
    verbose=-1
)

# The fix is in the line below: '_score_model' has been corrected to 'score_model'.
model = Treeffuser(score_model=gpboost_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_GPBoost_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_GPBoost_TF_df.head())


[1]	valid_0's l2: 1.12304
[2]	valid_0's l2: 1.04672
[3]	valid_0's l2: 1.00981
[4]	valid_0's l2: 0.946541
[5]	valid_0's l2: 0.883002
[6]	valid_0's l2: 0.853695
[7]	valid_0's l2: 0.823623
[8]	valid_0's l2: 0.799177
[9]	valid_0's l2: 0.763438
[10]	valid_0's l2: 0.737752
[11]	valid_0's l2: 0.743112
[12]	valid_0's l2: 0.7438
[13]	valid_0's l2: 0.735059
[14]	valid_0's l2: 0.739398
[15]	valid_0's l2: 0.722734
[16]	valid_0's l2: 0.744156
[17]	valid_0's l2: 0.732841
[18]	valid_0's l2: 0.706672


[19]	valid_0's l2: 0.68327
[20]	valid_0's l2: 0.706916
[21]	valid_0's l2: 0.692472
[22]	valid_0's l2: 0.698362
[23]	valid_0's l2: 0.697235
[24]	valid_0's l2: 0.698215
[25]	valid_0's l2: 0.703603
[26]	valid_0's l2: 0.680728
[27]	valid_0's l2: 0.668304
[28]	valid_0's l2: 0.659579
[29]	valid_0's l2: 0.65487
[30]	valid_0's l2: 0.659312
[31]	valid_0's l2: 0.661497
[32]	valid_0's l2: 0.647675
[33]	valid_0's l2: 0.639651
[34]	valid_0's l2: 0.629875
[35]	valid_0's l2: 0.633854
[36]	valid_0's l2: 0.614217
[37]	valid_0's l2: 0.612584
[38]	valid_0's l2: 0.606443
[39]	valid_0's l2: 0.597014
[40]	valid_0's l2: 0.592507
[41]	valid_0's l2: 0.595925
[42]	valid_0's l2: 0.586148
[43]	valid_0's l2: 0.586148
[44]	valid_0's l2: 0.584661


[45]	valid_0's l2: 0.588795
[46]	valid_0's l2: 0.573098
[47]	valid_0's l2: 0.557683
[48]	valid_0's l2: 0.555289
[49]	valid_0's l2: 0.549296
[50]	valid_0's l2: 0.547592
[51]	valid_0's l2: 0.537273
[52]	valid_0's l2: 0.535217
[53]	valid_0's l2: 0.536032
[54]	valid_0's l2: 0.534479
[55]	valid_0's l2: 0.524463
[56]	valid_0's l2: 0.516771
[57]	valid_0's l2: 0.522881
[58]	valid_0's l2: 0.515552
[59]	valid_0's l2: 0.515511
[60]	valid_0's l2: 0.521906
[61]	valid_0's l2: 0.531013
[62]	valid_0's l2: 0.532956
[63]	valid_0's l2: 0.53511
[64]	valid_0's l2: 0.539456


[65]	valid_0's l2: 0.540046
[66]	valid_0's l2: 0.536971
[67]	valid_0's l2: 0.527502
[68]	valid_0's l2: 0.529593
[69]	valid_0's l2: 0.529791
[70]	valid_0's l2: 0.528375
[71]	valid_0's l2: 0.52887
[72]	valid_0's l2: 0.528312
[73]	valid_0's l2: 0.530605
[74]	valid_0's l2: 0.52922
[75]	valid_0's l2: 0.528938
[76]	valid_0's l2: 0.529854
[77]	valid_0's l2: 0.52892
[78]	valid_0's l2: 0.533903
[79]	valid_0's l2: 0.528125
[80]	valid_0's l2: 0.528737
[81]	valid_0's l2: 0.52529
[82]	valid_0's l2: 0.516589
[83]	valid_0's l2: 0.520122
[84]	valid_0's l2: 0.528432
[85]	valid_0's l2: 0.529306
[86]	valid_0's l2: 0.536553
[87]	valid_0's l2: 0.537836
[88]	valid_0's l2: 0.54158
[89]	valid_0's l2: 0.543363
[90]	valid_0's l2: 0.53302
[91]	valid_0's l2: 0.532599


[92]	valid_0's l2: 0.530884
[93]	valid_0's l2: 0.534815
[94]	valid_0's l2: 0.534209
[95]	valid_0's l2: 0.53369
[96]	valid_0's l2: 0.533793
[97]	valid_0's l2: 0.536124
[98]	valid_0's l2: 0.538565
[99]	valid_0's l2: 0.539653
[100]	valid_0's l2: 0.536163
[101]	valid_0's l2: 0.534682
[102]	valid_0's l2: 0.534244
[103]	valid_0's l2: 0.534498
[104]	valid_0's l2: 0.539817
[105]	valid_0's l2: 0.539655
[106]	valid_0's l2: 0.537239
[107]	valid_0's l2: 0.538759
[108]	valid_0's l2: 0.538168
[109]	valid_0's l2: 0.538627


[110]	valid_0's l2: 0.538347
[111]	valid_0's l2: 0.533312
[112]	valid_0's l2: 0.534662
[113]	valid_0's l2: 0.535497
[114]	valid_0's l2: 0.538875
[115]	valid_0's l2: 0.541372
[116]	valid_0's l2: 0.543255
[117]	valid_0's l2: 0.546236
[118]	valid_0's l2: 0.545084
[119]	valid_0's l2: 0.543406
[120]	valid_0's l2: 0.541905
[121]	valid_0's l2: 0.543941
[122]	valid_0's l2: 0.545105
[123]	valid_0's l2: 0.544279
[124]	valid_0's l2: 0.541976
[125]	valid_0's l2: 0.542457
[126]	valid_0's l2: 0.542591
[127]	valid_0's l2: 0.542102
[128]	valid_0's l2: 0.539959
[129]	valid_0's l2: 0.537163
[130]	valid_0's l2: 0.538371
[131]	valid_0's l2: 0.539146
[132]	valid_0's l2: 0.541888
[133]	valid_0's l2: 0.542691
[134]	valid_0's l2: 0.540555
[135]	valid_0's l2: 0.545368
[136]	valid_0's l2: 0.542556
[137]	valid_0's l2: 0.542295
[138]	valid_0's l2: 0.541851


[139]	valid_0's l2: 0.541949
[140]	valid_0's l2: 0.538301
[141]	valid_0's l2: 0.546376
[142]	valid_0's l2: 0.546521
[143]	valid_0's l2: 0.540974
[144]	valid_0's l2: 0.547388
[145]	valid_0's l2: 0.553978
[146]	valid_0's l2: 0.554153
[147]	valid_0's l2: 0.555187
[148]	valid_0's l2: 0.546725
[149]	valid_0's l2: 0.549831
[150]	valid_0's l2: 0.548168
[151]	valid_0's l2: 0.547012
[152]	valid_0's l2: 0.550297
[153]	valid_0's l2: 0.549605
[154]	valid_0's l2: 0.547126
[155]	valid_0's l2: 0.547015
[156]	valid_0's l2: 0.547885


[157]	valid_0's l2: 0.550458
[158]	valid_0's l2: 0.548844
[159]	valid_0's l2: 0.549627
[160]	valid_0's l2: 0.55024
[161]	valid_0's l2: 0.548893
[162]	valid_0's l2: 0.548852
[163]	valid_0's l2: 0.550433
[164]	valid_0's l2: 0.549224
[165]	valid_0's l2: 0.548547
[166]	valid_0's l2: 0.547571
[167]	valid_0's l2: 0.547355
[168]	valid_0's l2: 0.547601
[169]	valid_0's l2: 0.546701
[170]	valid_0's l2: 0.546372
[171]	valid_0's l2: 0.547194
[172]	valid_0's l2: 0.548868
[173]	valid_0's l2: 0.548064
[174]	valid_0's l2: 0.548404
[175]	valid_0's l2: 0.548832
[176]	valid_0's l2: 0.548816
[177]	valid_0's l2: 0.550468
[178]	valid_0's l2: 0.54963
[179]	valid_0's l2: 0.552506
[180]	valid_0's l2: 0.551109
[181]	valid_0's l2: 0.551017
[182]	valid_0's l2: 0.55167
[183]	valid_0's l2: 0.551885
[184]	valid_0's l2: 0.552085
[185]	valid_0's l2: 0.552583


[186]	valid_0's l2: 0.552493
[187]	valid_0's l2: 0.551718
[188]	valid_0's l2: 0.552028
[189]	valid_0's l2: 0.551329
[190]	valid_0's l2: 0.553313
[191]	valid_0's l2: 0.553948
[192]	valid_0's l2: 0.549144
[193]	valid_0's l2: 0.548853
[194]	valid_0's l2: 0.549391
[195]	valid_0's l2: 0.550486
[196]	valid_0's l2: 0.550418
[197]	valid_0's l2: 0.550422
[198]	valid_0's l2: 0.548504
[199]	valid_0's l2: 0.548916
[200]	valid_0's l2: 0.545224


         Mean      StdDev  Actual
0  316.823766  110.845794     315
1  144.623293   19.768224     167
2  428.463669  126.271173     516
3  204.993859   30.275887     144
4  133.121276   12.641260     127


In [32]:
generate_and_save_plots(predictions_GPBoost_TF_df, y_test, "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx


In [33]:
generate_and_save_crps_decomposition(
    predictions_GPBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx",
    model_name="GPBoost",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx
   Sheets: CRPS_Summary_GPBoost, CRPS_Bins_GPBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,GPBoost,19.262531,2.490101,20.715405,37.490482,8


In [34]:
generate_and_save_pit_results(
    predictions_GPBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx",
    model_name="GPBoost",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx


,PIT_Value
0,0.493436
1,0.871173
2,0.755921
3,0.021973
4,0.314111
...,...
77,0.414888
78,0.682544
79,0.069062
80,0.248287


In [35]:
generate_and_save_calibration_curve(
    predictions_GPBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx",
    model_name="GPBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.109756,0.009756
2,0.2,0.243902,0.043902
3,0.3,0.365854,0.065854
4,0.4,0.524390,0.124390
5,0.5,0.682927,0.182927
6,0.6,0.817073,0.217073
7,0.7,0.914634,0.214634
8,0.8,0.951220,0.151220
9,0.9,1.000000,0.100000


# **Probabilistic Distribution using Gradient Boosting (Treeffuser)**

In [36]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')

gbr_score_engine = TreeBasedScoreModel(
    model_name='gradient_boosting',
    # FIX T1: the diffusion score model is fitted by least squares (Treeffuser's denoising
    # objective); the tuned GBR loss may be absolute_error/huber, so force squared_error here.
    model_params={**best_params, 'loss': 'squared_error'},
    seed=42,
)

model = Treeffuser(score_model=gbr_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_GBR_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_GBR_TF_df.head())


         Mean      StdDev  Actual
0  284.462406  126.670056     315
1  173.541193   70.598064     167
2  446.499296  108.956923     516
3  243.464329  105.625629     144
4  117.462066   73.516253     127


In [37]:
generate_and_save_plots(predictions_GBR_TF_df, y_test, "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx


In [38]:
generate_and_save_crps_decomposition(
    predictions_GBR_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx",
    model_name="Gradient Boosting",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx
   Sheets: CRPS_Summary_Gradient Boosting, CRPS_Bins_Gradient Boosting


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,Gradient Boosting,26.66883,10.380779,21.19808,37.490482,8


In [39]:
generate_and_save_pit_results(
    predictions_GBR_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx",
    model_name="GBoosting",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx


,PIT_Value
0,0.595253
1,0.463089
2,0.738222
3,0.173181
4,0.551614
...,...
77,0.400954
78,0.373776
79,0.195357
80,0.232096


In [40]:
generate_and_save_calibration_curve(
    predictions_GBR_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx",
    model_name="GBoosting",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.024390,0.175610
3,0.3,0.146341,0.153659
4,0.4,0.426829,0.026829
5,0.5,0.719512,0.219512
6,0.6,0.939024,0.339024
7,0.7,0.975610,0.275610
8,0.8,1.000000,0.200000
9,0.9,1.000000,0.100000


# **Probabilistic Distribution using Catboost (Treeffuser)**

In [41]:
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')

catboost_score_engine = TreeBasedScoreModel(
    model_name='catboost',
    model_params=best_params,
    seed=42,
)

model = Treeffuser(score_model=catboost_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_CatBoost_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_CatBoost_TF_df.head())

         Mean      StdDev  Actual
0  315.352035  271.503094     315
1  139.977943  129.825149     167
2  437.385781  220.934054     516
3  234.917715   78.667242     144
4  140.262151  374.861998     127


In [42]:
generate_and_save_plots(predictions_CatBoost_TF_df, y_test, "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx


In [43]:
generate_and_save_crps_decomposition(
    predictions_CatBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx",
    model_name="CatBoost",
    n_bins=N_BINS
)


✅ CRPS decomposition saved to:
drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx
   Sheets: CRPS_Summary_CatBoost, CRPS_Bins_CatBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,CatBoost,48.255079,25.553449,14.781042,37.490482,8


In [44]:
generate_and_save_pit_results(
    predictions_CatBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx",
    model_name="CatBoost",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx


,PIT_Value
0,0.499483
1,0.582441
2,0.639015
3,0.123897
4,0.485889
...,...
77,0.533737
78,0.504750
79,0.182428
80,0.417556


In [45]:
generate_and_save_calibration_curve(
    predictions_CatBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx",
    model_name="CatBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.024390,0.175610
3,0.3,0.060976,0.239024
4,0.4,0.268293,0.131707
5,0.5,0.695122,0.195122
6,0.6,0.951220,0.351220
7,0.7,1.000000,0.300000
8,0.8,1.000000,0.200000
9,0.9,1.000000,0.100000


# **Probabilistic Distribution using XGBoost (Treeffuser)**

In [46]:
best_params = get_best_model_params(best_scores_autosampler, 'XGBoost')

xgb_score_engine = TreeBasedScoreModel(
    model_name='xgboost',
    model_params=best_params,
    seed=42,
    verbose=-1
)

model = Treeffuser(score_model=xgb_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_XGB_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_XGB_TF_df.head())

[0]	validation_0-rmse:1.08387


[1]	validation_0-rmse:1.04961


[2]	validation_0-rmse:1.02805


[3]	validation_0-rmse:1.00409


[4]	validation_0-rmse:0.98235


[5]	validation_0-rmse:0.96188


... (491 outputs left out)

[497]	validation_0-rmse:0.72645

[498]	validation_0-rmse:0.72641


[499]	validation_0-rmse:0.72619


         Mean      StdDev  Actual
0  290.944594  286.560254     315
1  166.534018  132.121989     167
2  541.535540  448.109965     516
3  202.081880  113.521478     144
4  124.642723   34.913485     127


In [47]:
generate_and_save_plots(predictions_XGB_TF_df, y_test, "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx


In [48]:
generate_and_save_crps_decomposition(
    predictions_XGB_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx",
    model_name="XGBoost",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx
   Sheets: CRPS_Summary_XGBoost, CRPS_Bins_XGBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,XGBoost,29.301468,11.823592,20.007796,37.490482,8


In [49]:
generate_and_save_pit_results(
    predictions_XGB_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx",
    model_name="XGBoost",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx


,PIT_Value
0,0.533450
1,0.501407
2,0.477279
3,0.304452
4,0.526915
...,...
77,0.576846
78,0.512901
79,0.160025
80,0.353865


In [50]:
generate_and_save_calibration_curve(
    predictions_XGB_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx",
    model_name="XGBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.012195,0.087805
2,0.2,0.073171,0.126829
3,0.3,0.121951,0.178049
4,0.4,0.317073,0.082927
5,0.5,0.646341,0.146341
6,0.6,0.890244,0.290244
7,0.7,0.975610,0.275610
8,0.8,1.000000,0.200000
9,0.9,1.000000,0.100000


# **Probabilistic Distribution using LGBM (Treeffuser)**

In [51]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

lgbm_score_engine = TreeBasedScoreModel(
    model_name='lightgbm',
    model_params=best_params,
    seed=42,
    verbose=-1
)

model = Treeffuser(score_model=lgbm_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_LGBM_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_LGBM_TF_df.head())


         Mean      StdDev  Actual
0  296.448298  105.793006     315
1  158.880136  138.581932     167
2  476.221401  224.392362     516
3  240.713847  134.010912     144
4  128.091201   63.240515     127


In [52]:
generate_and_save_plots(predictions_LGBM_TF_df, y_test, "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx


In [53]:
generate_and_save_crps_decomposition(
    predictions_LGBM_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx",
    model_name="LightGBM",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx
   Sheets: CRPS_Summary_LightGBM, CRPS_Bins_LightGBM


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,LightGBM,35.765567,15.271013,16.990507,37.490482,8


In [54]:
generate_and_save_pit_results(
    predictions_LGBM_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx",
    model_name="LightGBM",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx


,PIT_Value
0,0.569601
1,0.523362
2,0.570353
3,0.235244
4,0.493117
...,...
77,0.497681
78,0.488534
79,0.204781
80,0.305693


In [55]:
generate_and_save_calibration_curve(
    predictions_LGBM_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx",
    model_name="LightGBM",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.024390,0.175610
3,0.3,0.121951,0.178049
4,0.4,0.292683,0.107317
5,0.5,0.707317,0.207317
6,0.6,0.975610,0.375610
7,0.7,1.000000,0.300000
8,0.8,1.000000,0.200000
9,0.9,1.000000,0.100000


# **IBUG Installation**

In [56]:
# Clone the repository
!git clone https://github.com/jjbrophy47/ibug.git
%cd ibug

# Build extensions if needed
!python setup.py build_ext --inplace

# Install the package
!pip install .

# Verify installation
try:
    from ibug import IBUGWrapper
    print("IBUGWrapper imported successfully!")
except ImportError as e:
    print("Error importing IBUGWrapper:", e)

Cloning into 'ibug'...


remote: Enumerating objects: 1763, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (31/31), done.


remote: Total 1763 (delta 18), reused 15 (delta 4), pack-reused 1726 (from 1)
Receiving objects: 100% (1763/1763), 1.31 MiB | 8.87 MiB/s, done.


Resolving deltas: 100% (967/967), done.


/kaggle/working/ibug


Compiling ibug/parsers/_tree32.pyx because it changed.
Compiling ibug/parsers/_tree64.pyx because it changed.
[1/2] Cythonizing ibug/parsers/_tree32.pyx


... (16 outputs left out)

  Preparing metadata (pyproject.toml) ... done


  Created wheel for ibug: filename=ibug-0.0.12-cp313-cp313-linux_x86_64.whl size=1581581 sha256=25dfbbe5eb6033cfc906286ec480b2bbdb968957550daf67d6a99da5833e7893
  Stored in directory: /tmp/pip-ephem-wheel-cache-qee8bhr9/wheels/48/c2/c1/d39bc91440bc7e7da40b146580d8aeb84c5361fa857fe8037e
Successfully built ibug


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ibug]


IBUGWrapper imported successfully!


In [57]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=42)  # FIX P2: seeded


# **Probabilistic Distributions using LGBM (IBUG)**

In [58]:
# y_train = y_train.to_numpy()
# y_val = y_val.to_numpy()

In [59]:
# Assume hyperparameter tuning has been done and best_params are obtained
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

# --- Ensure IBUG compatibility ---
best_params['reg_alpha'] = 0
best_params['class_weight'] = None
best_params['boosting_type'] = 'gbdt'

# Initialize and train the LightGBM regressor using the best_params
model = LGBMRegressor(**best_params)
model.fit(X_train, y_train)

# Extend the trained model into a probabilistic predictor using IBUGWrapper
prob_model = IBUGWrapper().fit(model, X_train, y_train, X_val=X_val, y_val=y_val)

# Use the model to predict the probabilistic distribution on the test data.
location, scale = prob_model.pred_dist(X_test)

# Optionally, to also get the k-nearest training indices and their values used in modeling
location2, scale2, train_idxs, train_vals = prob_model.pred_dist(X_test, return_kneighbors=True)

dev = scale  # FIX P2: IBUG's scale is already a standard deviation (np.std of neighbours); was scale**0.5
# Create a DataFrame for clarity
predictions_LGBM_df = pd.DataFrame({
    'Mean': location,
    'StdDev': dev,
    'Actual': y_test,
})

print(predictions_LGBM_df.head())

         Mean     StdDev  Actual
0  289.326540  39.485241     315
1  185.157299  52.249138     167
2  380.806760  57.730141     516
3  204.276102  35.576805     144
4  127.877822  20.578733     127


In [60]:
%cd ..

/kaggle/working


In [61]:
%cd ..

/kaggle


In [62]:
generate_and_save_plots(predictions_LGBM_df, y_test, "./content/drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx")

Excel file saved at ./content/drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx


In [63]:
generate_and_save_crps_decomposition(
    predictions_LGBM_df,
    y_test,
    "/content/drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx",
    model_name="LGBM",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
/content/drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx
   Sheets: CRPS_Summary_LGBM, CRPS_Bins_LGBM


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,LGBM,16.318442,2.974635,24.145002,37.490482,8


In [64]:
generate_and_save_pit_results(
    predictions_LGBM_df,
    y_test,
    "/content/drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx",
    model_name="LGBM",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   /content/drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx


,PIT_Value
0,0.742220
1,0.364103
2,0.990405
3,0.045109
4,0.482988
...,...
77,0.448383
78,0.304004
79,0.155475
80,0.162601


In [65]:
generate_and_save_calibration_curve(
    predictions_LGBM_df,
    y_test,
    "/content/drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx",
    model_name="LGBM",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   /content/drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.097561,0.002439
2,0.2,0.231707,0.031707
3,0.3,0.426829,0.126829
4,0.4,0.609756,0.209756
5,0.5,0.792683,0.292683
6,0.6,0.890244,0.290244
7,0.7,0.951220,0.251220
8,0.8,0.975610,0.175610
9,0.9,0.987805,0.087805


# **Probabilistic Distributions using XGBoost (IBUG)**

In [66]:
# Assume hyperparameter tuning has been done and best_params are obtained
best_params = get_best_model_params(best_scores_autosampler, 'XGBoost')

best_params['reg_alpha'] = 0
best_params['scale_pos_weight'] = 1
best_params['base_score'] = 0.5

# Initialize and train the XGBoost regressor using the best_params
model = XGBRegressor(**best_params)
model.fit(X_train, y_train)

# Extend the trained model into a probabilistic predictor using IBUGWrapper
prob_model = IBUGWrapper().fit(model, X_train, y_train, X_val=X_val, y_val=y_val)

# Use the model to predict the probabilistic distribution on the test data.
# This returns the predicted mean (location) and uncertainty (scale, variance proxy)
location, scale = prob_model.pred_dist(X_test)

# Optionally, to also get the k-nearest training indices and their values used in modeling
location, scale, train_idxs, train_vals = prob_model.pred_dist(X_test, return_kneighbors=True)

# Calculate the standard deviation from the variance (scale)
std_dev = scale  # FIX P2: IBUG's scale is already a standard deviation (np.std of neighbours); was scale ** 0.5

# Create a DataFrame for clarity
predictions_XGB_df = pd.DataFrame({
    'Mean': location,
    'StdDev': std_dev,
    'Actual': y_test
})

print(predictions_XGB_df.head())

         Mean     StdDev  Actual
0  294.316376  10.183805     315
1  172.612930  22.947336     167
2  416.970703  33.964031     516
3  231.166199  42.563316     144
4  145.282394   9.332672     127


In [67]:
generate_and_save_plots(predictions_XGB_df, y_test, "./content/drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx")  # FIX P3: was predictions_LGBM_df

Excel file saved at ./content/drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx


In [68]:
generate_and_save_crps_decomposition(
    predictions_XGB_df,
    y_test,
    "/content/drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx",
    model_name="XGB",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
/content/drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx
   Sheets: CRPS_Summary_XGB, CRPS_Bins_XGB


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,XGB,14.957795,3.018182,25.548385,37.490482,8


In [69]:
generate_and_save_pit_results(
    predictions_XGB_df,
    y_test,
    "/content/drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx",
    model_name="XGB",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   /content/drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx


,PIT_Value
0,0.978874
1,0.403383
2,0.998226
3,0.020284
4,0.025058
...,...
77,0.352537
78,0.255292
79,0.002817
80,0.023681


In [70]:
generate_and_save_calibration_curve(
    predictions_XGB_df,
    y_test,
    "/content/drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx",
    model_name="XGB",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   /content/drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.365854,0.265854
2,0.2,0.500000,0.300000
3,0.3,0.585366,0.285366
4,0.4,0.634146,0.234146
5,0.5,0.731707,0.231707
6,0.6,0.865854,0.265854
7,0.7,0.902439,0.202439
8,0.8,0.914634,0.114634
9,0.9,0.939024,0.039024


# **Matrix Evaulation**

In [71]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

# Create a Pandas Excel writer using XlsxWriter as the engine.
folder_path = "/content/drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis"
excel_path = os.path.join(folder_path, "/content/drive/MyDrive/Regression/water_quality_wqi_uncertainty_analysis/probabilistic_distribution/Matrix Evaluation.xlsx")
with pd.ExcelWriter(_ensure_parent_dir(excel_path), engine='xlsxwriter') as writer:
    plot_metrics_and_save_to_excel(writer, predictions_NGB_df, y_test, 'NGBoost')
    plot_metrics_and_save_to_excel(writer, predictions_PGBM_df, y_test, 'PGBM')
    plot_metrics_and_save_to_excel(writer, predictions_LGBM_df, y_test, 'LGBM')
    plot_metrics_and_save_to_excel(writer, predictions_XGB_df, y_test, 'XGBoost')
    plot_metrics_and_save_to_excel(writer, predictions_LGBM_TF_df, y_test, "LightGBM_TF")
    plot_metrics_and_save_to_excel(writer, predictions_XGB_TF_df, y_test, "XGBoost_TF")
    plot_metrics_and_save_to_excel(writer, predictions_CatBoost_TF_df, y_test, "CatBoost_TF")
    plot_metrics_and_save_to_excel(writer, predictions_GPBoost_TF_df, y_test, "GPBoost_TF")
    plot_metrics_and_save_to_excel(writer, predictions_GBR_TF_df, y_test, "GradientBoosting_TF")